<a href="https://colab.research.google.com/github/Vedish-18/safe-paws-app/blob/main/Electricity_Consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import zipfile
import os

# Download the UCI Household Power Consumption dataset
url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
zip_path = "dataset.zip"
txt_file = "household_power_consumption.txt"

if not os.path.exists(txt_file):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete!")
else:
    print(f"'{txt_file}' already exists.")

In [ ]:
import pandas as pd
import numpy as np

print("Loading raw telemetry data (this may take 20-30 seconds)...")
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    parse_dates={"timestamp": ["Date", "Time"]},
    dayfirst=True,
    na_values=["?"],
    low_memory=False
)

df.set_index("timestamp", inplace=True)
df.sort_index(inplace=True)

# Impute short sensor dropouts
df = df.ffill().bfill()

# Downsample from minute-level to hourly averages/sums
print("Resampling to hourly intervals...")
hourly_df = df.resample("1h").agg({
    "Global_active_power": "mean",
    "Global_reactive_power": "mean",
    "Voltage": "mean",
    "Global_intensity": "mean",
    "Sub_metering_1": "sum",
    "Sub_metering_2": "sum",
    "Sub_metering_3": "sum"
})

hourly_df.dropna(inplace=True)
print(f"Dataset ready. Total hourly rows: {len(hourly_df)}")
hourly_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Identify top 5% peak demand hours (grid stress events)
peak_threshold = hourly_df["Global_active_power"].quantile(0.95)
hourly_df["is_peak"] = hourly_df["Global_active_power"] >= peak_threshold

print(f"--- SDG 7 Peak Demand Profiling ---")
print(f"95th Percentile Peak Cutoff: {peak_threshold:.3f} kW")
print(f"Total Peak Stress Hours: {hourly_df['is_peak'].sum()} hrs")

hourly_df["Hour"] = hourly_df.index.hour
hourly_df["DayOfWeek"] = hourly_df.index.day_name()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Average Diurnal Curves
sns.lineplot(
    data=hourly_df,
    x="Hour",
    y="Global_active_power",
    hue="DayOfWeek",
    errorbar=None,
    ax=axes[0]
)
axes[0].axhline(peak_threshold, color="crimson", linestyle="--", label=f"Peak Limit ({peak_threshold:.2f} kW)")
axes[0].set_title("Diurnal Energy Profile by Day of Week")
axes[0].set_ylabel("Demand (kW)")
axes[0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

# 2. Load Duration Curve
sorted_load = np.sort(hourly_df["Global_active_power"].values)[::-1]
axes[1].plot(sorted_load, color="navy")
axes[1].axhline(peak_threshold, color="crimson", linestyle="--")
axes[1].set_title("Load Duration Curve")
axes[1].set_xlabel("Ranked Hours (High to Low Demand)")
axes[1].set_ylabel("Demand (kW)")

plt.tight_layout()
plt.show()

In [ ]:
data = hourly_df.copy()
target = "Global_active_power"

# Cyclical time encodings
data["sin_hour"] = np.sin(2 * np.pi * data["Hour"] / 24.0)
data["cos_hour"] = np.cos(2 * np.pi * data["Hour"] / 24.0)
data["sin_month"] = np.sin(2 * np.pi * data.index.month / 12.0)
data["cos_month"] = np.cos(2 * np.pi * data.index.month / 12.0)
data["dayofweek"] = data.index.dayofweek

# Lag features
data["lag_1"] = data[target].shift(1)
data["lag_2"] = data[target].shift(2)
data["lag_24"] = data[target].shift(24)    # Same hour yesterday
data["lag_168"] = data[target].shift(168)  # Same hour last week

# Rolling window statistics (shifted by 1 to prevent data leakage)
data["rolling_mean_6h"] = data[target].shift(1).rolling(6).mean()
data["rolling_std_6h"] = data[target].shift(1).rolling(6).std()
data["rolling_mean_24h"] = data[target].shift(1).rolling(24).mean()
data["rolling_std_24h"] = data[target].shift(1).rolling(24).std()

data.dropna(inplace=True)
print(f"Feature dataset shape: {data.shape}")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

feature_cols = [
    "sin_hour", "cos_hour", "sin_month", "cos_month", "dayofweek",
    "lag_1", "lag_2", "lag_24", "lag_168",
    "rolling_mean_6h", "rolling_std_6h", "rolling_mean_24h", "rolling_std_24h",
    "Global_reactive_power", "Voltage", "Global_intensity"
]

# Chronological Train-Test Split (Final 3 months reserved as unseen test set)
split_date = data.index.max() - pd.DateOffset(months=3)
train = data.loc[data.index < split_date]
test = data.loc[data.index >= split_date]

X_train, y_train = train[feature_cols], train[target]
X_test, y_test = test[feature_cols], test[target]

print(f"Training observations: {len(X_train)} | Test observations: {len(X_test)}")

# Train model with early stopping
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# Predictions & Metrics
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds) * 100

# Evaluate model performance specifically during peak demand events
peak_limit = y_test.quantile(0.95)
peak_mask = y_test >= peak_limit
peak_mae = mean_absolute_error(y_test[peak_mask], preds[peak_mask])

print("\n==== OUT-OF-SAMPLE EVALUATION ====")
print(f"MAE  : {mae:.4f} kW")
print(f"RMSE : {rmse:.4f} kW")
print(f"MAPE : {mape:.2f}%")
print(f"Peak Demand MAE (Top 5% spikes): {peak_mae:.4f} kW")

# Plot Last 7 Days of Test Horizon
plt.figure(figsize=(14, 5))
plt.plot(y_test.index[-168:], y_test.iloc[-168:], label="Actual Demand (kW)", color="black")
plt.plot(y_test.index[-168:], preds[-168:], label="LightGBM Prediction", color="crimson", linestyle="--")
plt.title("Household Electricity Consumption Prediction - Last 7 Days")
plt.xlabel("Timestamp")
plt.ylabel("Active Power (kW)")
plt.legend()
plt.tight_layout()
plt.show()